In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

from functools import partial

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
# Define the action
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

# Component actions for testing composition
class ScalarKineticAction(lhmc.Action):
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2

        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        return S

class ScalarQuarticInt(lhmc.Action):    
    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        return params['lambda'] * (phi**2 - 1)**2




In [3]:
# Actions depend on parameters, fields are arbitrary including dimension
S_A = ScalarAction(params = {'kappa': 0.18, 'lambda': 1.0}, field_names=['phi'])
S_B = ScalarAction(params = {'kappa': 0.17, 'lambda': 1.0}, field_names=['phi'])

# Force calculation should work automatically, without explicit definition above
F_A = S_A.get_forces()
print(F_A)


# Calling the action object as a function, using fields as input, should call and return the summed action functional
L4 = lat.SquareLattice(dims=(4,4))
L6 = lat.SquareLattice(dims=(6,6))
phi4 = lat.LatticeField(L4).unit_fill()
phi6 = lat.LatticeField(L6).unit_fill()
SA_total_4 = S_A.S({'phi': phi4})
SA_total_6 = S_A.S({'phi': phi6})
SB_total_4 = S_B.S({'phi': phi4})

print(SA_total_4, SA_total_6, SB_total_4)

<PjitFunction of <function Action._compute_forces.<locals>.<lambda> at 0x142c920c0>>
4.48 10.08 5.119999999999998


I0000 00:00:1703515375.498619       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [4]:
S_A.S({'phi': phi4, 'phi2': phi4})

Array(4.48, dtype=float64)

In [7]:
%timeit S_A.S({'phi': phi4})

13.7 µs ± 68.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [8]:
print(F_A({'phi': phi4})['phi'].F)
print(S_A.exact_force(fields=[phi4], params=S_A.params).F)

%timeit F_A({'phi': phi4})
%timeit S_A.exact_force(fields=[phi4], params=S_A.params)

[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
[[0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]
 [0.56 0.56 0.56 0.56]]
5.46 µs ± 39.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
7.72 µs ± 75.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [9]:
# We should be able to build a joint action with *shared* fields, by composition.
S_A_kin = ScalarKineticAction(params={'kappa': 0.18}, field_names=['chi'])
S_A_int = ScalarQuarticInt(params={'lambda': 1.0}, field_names=['chi'])

S_A_joint = S_A_kin + S_A_int
print(S_A_joint.S({'chi': phi4}))  # Should match cell above
      
# On the other hand, we should also be able to compose multiple copies of a given action,
# without making the fields shared.
S_A_different = ScalarAction(params={'kappa': 0.18, 'lambda': 1.0}, field_names=['phi2'])
S_A_disjoint = S_A + S_A_different

print(S_A_disjoint.S(fields={'phi':phi4, 'phi2': phi4}))  # Should be 2x the action above

4.48
8.96


In [10]:
# Some edge cases/error modes:

# Trying to combine actions with two different field dimensions should error
try:
    print(S_A_disjoint.S(fields={'phi': phi4, 'phi2': phi6}))
except Exception as e:
    print(e)


add got incompatible shapes for broadcasting: (4, 4), (6, 6).
